In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, lit

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6b3ffcac-9524-4713-af35-695a85461dc1;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 339ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
tb_orders = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-orders/final_table_orders_numeric.csv/", header=True, inferSchema=True)
tb_dates = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-holidays/dates_2016_2018/", header=True, inferSchema=True)

26/06/16 00:10:11 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================================= =
# 1. PREPARAÇÃO DOS DADOS DE ENTRADA
# ============================================================================= =

# Ajustar o formato da tabela de feriados (ex: "dd/MM/yyyy")
tb_dates_prep = tb_dates.withColumn("data_feriado", F.to_date(F.col("date"), "dd/MM/yyyy"))

# ALTERAÇÃO AQUI: Extrair apenas a data de COMPRA dos pedidos (ao invés do envio)
tb_orders_prep = tb_orders.withColumn(
    "purchase_date_only", 
    F.to_date(F.col("order_purchase_timestamp"))
)

# ============================================================================= =
# 2. CRIAÇÃO DO CALENDÁRIO CONTÍNUO (Para evitar Joins repetitivos/pesados)
# ============================================================================= =

# Identificar o período total do seu dataset baseado na data de compra para gerar o calendário
date_range = tb_orders_prep.select(
    F.min("purchase_date_only").alias("min_date"), 
    F.max("purchase_date_only").alias("max_date")
).first()

# Em vez de pegar o min/max das compras, force uma janela maior de feriados:
start_date = F.to_date(F.lit("2015-01-01"))
end_date = F.to_date(F.lit("2019-12-31"))

# Gerando a sequência contínua de dias no PySpark
calendar_df = spark.range(1).select(
    F.explode(F.sequence(start_date, end_date, F.expr("interval 1 day"))).alias("calendar_date")
)

# Cruzando o calendário contínuo com os seus feriados reais
calendar_with_holidays = calendar_df.join(
    tb_dates_prep, \
    calendar_df.calendar_date == tb_dates_prep.data_feriado, \
    "left"
).withColumn(
    "is_holiday_today", 
    F.when(F.col("data_feriado").isNotNull(), 1).otherwise(0)
).select("calendar_date", "is_holiday_today")

# ============================================================================= =
# 3. CÁLCULO DAS JANELAS TEMPORAIS (Window Functions)
# ============================================================================= =

# Janela para olhar para TRÁS (Histórico: do início até o dia atual)
window_past = Window.orderBy("calendar_date").rowsBetween(Window.unboundedPreceding, 0)

# Janela para olhar para FRENTE (Futuro: do dia atual até o fim dos tempos)
window_future = Window.orderBy("calendar_date").rowsBetween(0, Window.unboundedFollowing)

# Criando colunas de apoio contendo a data do último e do próximo feriado
calendar_features = calendar_with_holidays.withColumn(
    "last_holiday_date", 
    F.max(F.when(F.col("is_holiday_today") == 1, F.col("calendar_date"))).over(window_past)
).withColumn(
    "next_holiday_date", \
    F.min(F.when(F.col("is_holiday_today") == 1, F.col("calendar_date"))).over(window_future)
)

# Calculando a distância exata em dias (Substitui nulos por valores altos/seguros)
calendar_features = calendar_features.withColumn(
    "days_since_last_holiday",
    F.coalesce(F.datediff(F.col("calendar_date"), F.col("last_holiday_date")), F.lit(999))
).withColumn(
    "days_until_next_holiday",
    F.coalesce(F.datediff(F.col("next_holiday_date"), F.col("calendar_date")), F.lit(999))
)

# Criando os sinalizadores binários de 7 e 14 dias para o futuro e passado
calendar_features = calendar_features.withColumn(
    "is_holiday_in_7_days", F.when(F.col("days_until_next_holiday") <= 7, 1).otherwise(0)
).withColumn(
    "is_holiday_in_14_days", F.when(F.col("days_until_next_holiday") <= 14, 1).otherwise(0)
).withColumn(
    "had_holiday_7_days_ago", F.when(F.col("days_since_last_holiday") <= 7, 1).otherwise(0)
).withColumn(
    "had_holiday_14_days_ago", F.when(F.col("days_since_last_holiday") <= 14, 1).otherwise(0)
)

# Selecionando apenas o que importa da nossa tabela dimensional de tempo
calendar_final_features = calendar_features.select(
    F.col("calendar_date").alias("match_date"),
    "is_holiday_today",
    "days_since_last_holiday",
    "days_until_next_holiday",
    "is_holiday_in_7_days",
    "is_holiday_in_14_days",
    "had_holiday_7_days_ago",
    "had_holiday_14_days_ago"
)

# ============================================================================= =
# 4. JOIN FINAL COM A TABELA DE PEDIDOS E SELEÇÃO DE COLUNAS
# ============================================================================= =

# ALTERAÇÃO AQUI: O Join agora é feito usando a 'purchase_date_only'
df_final = tb_orders_prep.join(
    calendar_final_features, 
    tb_orders_prep.purchase_date_only == calendar_final_features.match_date, 
    "left"
).withColumn(
    "holiday_at_purchase", F.col("is_holiday_today")  # Renomeado de 'holiday_at_shipping' para fazer sentido
)

# Organizando e garantindo todas as suas colunas numéricas originais + as novas features
# Substituí o 'holiday_at_shipping' por 'holiday_at_purchase' mantendo a mesma ordem
df_final = df_final.select(
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date", 
    "interval_code_delivered_carrier", \
    "order_delivered_customer_date", 
    "interval_code_delivered_customer",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "holiday_at_purchase",         # Modificado para refletir a data de compra
    "days_since_last_holiday",    # Nova coluna numérica (referente à compra)
    "days_until_next_holiday",    # Nova coluna numérica (referente à compra)
    "is_holiday_in_7_days",       # Nova coluna binária (referente à compra)
    "is_holiday_in_14_days",      # Nova coluna binária (referente à compra)
    "had_holiday_7_days_ago",     # Nova coluna binária (referente à compra)
    "had_holiday_14_days_ago",    # Nova coluna binária (referente à compra)
    "purchase_hour", 
    "purchase_day_of_week", \
    "is_weekend", 
    "delivered_on_time",
    "delivery_time_days",
    "price",
    "freight_value", 
    "product_weight_g", 
    "volume_cm3", \
    "distance_km",
    "category_name",
    "customer_city",
    "seller_city",
    "same_city", 
    "review_score"
).dropna(subset=["product_weight_g", "volume_cm3"])

# Exibir para conferência das novas colunas baseadas na data de compra
df_final.show(5)

26/06/16 00:10:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:31 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/16 00:10:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradat

+------------------------+-------------------+----------------------------+-------------------------------+-----------------------------+--------------------------------+-----------------------------+-------------------+-------------------+-----------------------+-----------------------+--------------------+---------------------+----------------------+-----------------------+-------------+--------------------+----------+-----------------+------------------+------+-------------+----------------+----------+-----------+---------------+---------------+-----------+---------+------------+
|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|interval_code_delivered_carrier|order_delivered_customer_date|interval_code_delivered_customer|order_estimated_delivery_date|shipping_limit_date|holiday_at_purchase|days_since_last_holiday|days_until_next_holiday|is_holiday_in_7_days|is_holiday_in_14_days|had_holiday_7_days_ago|had_holiday_14_days_ago|purchase_hour|purchase_day_of_we

In [5]:
df_final.groupBy("holiday_at_purchase").count().show()

+-------------------+-----+
|holiday_at_purchase|count|
+-------------------+-----+
|                  1|  948|
|                  0|32872|
+-------------------+-----+



In [6]:
# Filtra apenas onde é feriado e mostra as primeiras 20 linhas
df_final.filter(F.col("holiday_at_purchase") == 1).show()

26/06/16 00:10:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 0

+------------------------+-------------------+----------------------------+-------------------------------+-----------------------------+--------------------------------+-----------------------------+-------------------+-------------------+-----------------------+-----------------------+--------------------+---------------------+----------------------+-----------------------+-------------+--------------------+----------+-----------------+------------------+-----+-------------+----------------+----------+-----------+---------------+-------------+--------------------+---------+------------+
|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|interval_code_delivered_carrier|order_delivered_customer_date|interval_code_delivered_customer|order_estimated_delivery_date|shipping_limit_date|holiday_at_purchase|days_since_last_holiday|days_until_next_holiday|is_holiday_in_7_days|is_holiday_in_14_days|had_holiday_7_days_ago|had_holiday_14_days_ago|purchase_hour|purchase_day

In [7]:
from pyspark.sql import functions as F

# Versão limpa: checa o que é nulo (funciona para TODAS as colunas)
null_counts = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_final.columns
]

# Exibe o resultado das contagens
df_final.select(null_counts).show()

26/06/16 00:10:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 0

+------------------------+-----------------+----------------------------+-------------------------------+-----------------------------+--------------------------------+-----------------------------+-------------------+-------------------+-----------------------+-----------------------+--------------------+---------------------+----------------------+-----------------------+-------------+--------------------+----------+-----------------+------------------+-----+-------------+----------------+----------+-----------+-------------+-------------+-----------+---------+------------+
|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|interval_code_delivered_carrier|order_delivered_customer_date|interval_code_delivered_customer|order_estimated_delivery_date|shipping_limit_date|holiday_at_purchase|days_since_last_holiday|days_until_next_holiday|is_holiday_in_7_days|is_holiday_in_14_days|had_holiday_7_days_ago|had_holiday_14_days_ago|purchase_hour|purchase_day_of_week|is_wee

In [8]:
df_final.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-client/table_orders_holidays.csv')

spark.stop()

26/06/16 00:10:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 00:10:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 0